# Step 2: Multi-Label Encoding of Hand-Labeled Annotation Data

This notebook utilises the pilot annotation sheet , ('annotation_sheet.xlsx`), with hand-labeled data that will be used to train the model. The notebook aims to do the following:

1. Loading the hand-labeled subset (51 user reviews with at least one label filled in).
2. Removing reviews that were flagged as **not mental-health related**, which includes reviewes dealing with non-mental health apps like add blockers.
3. Adding a new **6th risk category, `user_complaint`**, in addition to the five other categories described int he project description. This category includes reviews that were flagged as general product complaints (e.g. cost, missing features) rather than a clinical/emotional risk signal.
4. Building **multi-label** targets: a review can belong to more than one category-- so every label marked `Yes` is saved.
5. One-hot / multi-hot encoding both the labels (targets) and `app_name` (a categorical feature) into numeric columns the classifier can train on.

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
#load annotation sheet
ANNOTATION_PATH = "../data/annotation_sheet.xlsx"

df_raw = pd.read_excel(ANNOTATION_PATH, sheet_name="pilot_sheet.csv")
df_raw.head()

,UID,app_name,rating,review,dependence,attachment,isolation/detachment,crisis-response frustration,none,other_risk_text,dominant,notes
0,151332,daylio,5,Great customisable app - highly recommend this...,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN
1,136235,digitalwellbeing,3,"Great feature for monitoring activity, I need ...",NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN
2,57666,sevencups,1,The people are not properly trained and their ...,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN
3,186996,minddoc,4,It's great app 😊,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN
4,106256,finch,5,honestly when I downloaded it I thought it'd j...,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN


In [4]:
# pull out labelled reviews
LABEL_COLS = [
    "dependence",
    "attachment",
    "isolation/detachment",
    "crisis-response frustration",
    "none",
]

is_labeled = df_raw[LABEL_COLS].notna().any(axis=1)
df_labeled = df_raw.loc[is_labeled].reset_index(drop=True)

# Keep the annotation order (review #1 = first labeled row, etc.) 
df_labeled.insert(0, "review_num", range(1, len(df_labeled) + 1))

print(f"Hand-labeled reviews found: {len(df_labeled)}")
df_labeled[["review_num", "UID", "app_name", "rating"] + LABEL_COLS].head(10)

Hand-labeled reviews found: 50


,review_num,UID,app_name,rating,dependence,attachment,isolation/detachment,crisis-response frustration,none
0,1,151332,daylio,5,NaN,NaN,NaN,NaN,Yes
1,2,136235,digitalwellbeing,3,NaN,NaN,NaN,NaN,Yes
2,3,57666,sevencups,1,NaN,NaN,NaN,Yes,NaN
3,4,186996,minddoc,4,NaN,NaN,NaN,NaN,Yes
4,5,106256,finch,5,NaN,NaN,NaN,NaN,Yes
5,6,8190,intellect,5,NaN,NaN,NaN,NaN,Yes
6,7,55625,sevencups,2,NaN,NaN,NaN,Yes,NaN
7,8,194762,wysa,3,NaN,NaN,Yes,Yes,NaN
8,9,148560,digitalwellbeing,1,NaN,NaN,NaN,NaN,Yes
9,10,182452,minddoc,5,Yes,NaN,NaN,NaN,NaN


In [6]:
#remove rows that are not mental-health related
#Reviews **#5, #15, #17, #20, #23, were flagged as not actually being about mental health, they include things liek reviews about app-blockers, so are removed from the dataset
# We match on `UID` (not row position) so this stays correct even if the sheet gets re-sorted, filtered, or more rows get added above these.
REMOVE_REVIEW_NUMS = [4, 14, 16, 19, 22]

removed_uids = df_labeled.loc[
    df_labeled["review_num"].isin(REMOVE_REVIEW_NUMS), "UID"
].tolist()
print("UIDs being removed:", removed_uids)

df_labeled = df_labeled.loc[~df_labeled["UID"].isin(removed_uids)].reset_index(drop=True)
print(f"Remaining hand-labeled reviews after removal: {len(df_labeled)}")

UIDs being removed: [186996, 77154, 137054, 200692, 80674]
Remaining hand-labeled reviews after removal: 39


In [7]:
## add the new `user_complaint` category
## Reviews **#3, #18, #24, #25, #32, #40, #48** describe general product complaints (billing, missing features, app quality, etc.) rather than — or in addition to — a clinical risk,
## so we add a **new** `user_complaint` label for them. 
## this is additive: whatever category(ies) a review already had stay, `user_complaint` is just flagged as well

COMPLAINT_REVIEW_NUMS = [47, 39, 31, 24, 23, 17, 2, 50] 

df_labeled["user_complaint"] = np.where(
    df_labeled["review_num"].isin(COMPLAINT_REVIEW_NUMS), "Yes", None
)

df_labeled.loc[
    df_labeled["review_num"].isin(COMPLAINT_REVIEW_NUMS),
    ["review_num", "UID", "app_name", "review"] + LABEL_COLS + ["user_complaint"],
]

,review_num,UID,app_name,review,dependence,attachment,isolation/detachment,crisis-response frustration,none,user_complaint
1,2,136235,digitalwellbeing,"Great feature for monitoring activity, I need ...",NaN,NaN,NaN,NaN,Yes,Yes
12,24,35979,dailybean,"Great app, but I wish it was an offline app.",NaN,NaN,NaN,NaN,Yes,Yes
19,31,39072,teencounseling,As bad as this app was I'm surprised I didn't ...,NaN,NaN,NaN,NaN,Yes,Yes
27,39,160962,daylio,"literally love this app so much, the only one ...",Yes,NaN,NaN,NaN,NaN,Yes
35,47,60049,sevencups,"Other than the very few lags, this app is perf...",Yes,NaN,NaN,NaN,NaN,Yes
38,50,2328,intellect,"It's not a bad app, but it's not for me. I app...",NaN,NaN,NaN,Yes,NaN,Yes


In [8]:
## build multi-label targets
## for eevry review extract all categories/labels that were marked as yes, so one review may be classified into multiple categories/labels

ALL_CATEGORY_COLS = [
    "dependence",
    "attachment",
    "isolation/detachment",
    "crisis-response frustration",
    "user_complaint",
    "none",
]

def get_labels(row):
    return [col for col in ALL_CATEGORY_COLS if row[col] == "Yes"]

df_labeled["labels"] = df_labeled.apply(get_labels, axis=1)
df_labeled[["review_num", "UID", "labels"]].head(10)

,review_num,UID,labels
0,1,151332,[none]
1,2,136235,"[user_complaint, none]"
2,3,57666,[crisis-response frustration]
3,7,55625,[crisis-response frustration]
4,8,194762,"[isolation/detachment, crisis-response frustra..."
5,9,148560,[none]
6,10,182452,[dependence]
7,11,55397,"[attachment, isolation/detachment]"
8,12,17487,[crisis-response frustration]
9,13,55470,[crisis-response frustration]


In [ ]:
##one-hot encode categorical features
## becuase the data is multi-labeled, pd.get_dummies doe snot work here so multilabelbinariser is used instead
mlb = MultiLabelBinarizer(classes=[
    "dependence",
    "attachment",
    "isolation/detachment",
    "crisis-response frustration",
    "user_complaint",
    "none",
])

label_matrix = mlb.fit_transform(df_labeled["labels"])
df_label_encoded = pd.DataFrame(label_matrix, columns=mlb.classes_, index=df_labeled.index)

df_label_encoded.head(10)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

## one-hot encode the name
## since 
ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)
app_encoded = ohe.fit_transform(df_labeled[["app_name"]])
df_app_encoded = pd.DataFrame(
    app_encoded, columns=ohe.get_feature_names_out(["app_name"]), index=df_labeled.index
)

df_app_encoded.head(10)